# 免费历史一致预期来源可行性

固定2022年三公司全部12次财报，先验证日历覆盖、普通预期与SmartEstimate列和文件时间。采用原始I/B/E/S简报，不混用媒体供应商。仅来源资格，不看收益、不训练。依赖pypdf6.20.1。

In [1]:
from pathlib import Path
import json,hashlib,os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import qlib,pandas as pd,yaml
from qlib.data import D
from qlib.workflow import R
from src.data.refinitiv_newsletter import collect_newsletters
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/refinitiv_newsletter_source_20261013';plan=json.loads((card/'plan.json').read_text())
for path,digest in plan['calendar_source_hashes'].items():assert hashlib.sha256((root/path).read_bytes()).hexdigest()==digest
qlib.init(provider_uri=str(root/plan['price_provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(freq='1min')).normalize().unique().sort_values()
assert all(pd.Timestamp(e['report_date']) in calendar for e in plan['events'])
raw=root/plan['raw_cache'];manifest=collect_newsletters(raw,plan);events=pd.DataFrame(manifest['entries'])
audit={'expected_events':len(plan['events']),'parsed_events':len(events),'failures':manifest['failures'],'per_symbol':events.groupby('symbol').size().to_dict() if len(events) else {},'prior_day_document_clock_rows':int(events.document_clock_before_report_day.sum()) if len(events) else 0,'transport_certificate_verified_rows':int(events.tls_certificate_verified.sum()) if len(events) else 0,'publication_clock_certified':False,'point_in_time_consensus_certified':False,'model_fits':0,'backtests':0}
issuer_raw=(raw/'nvda_20220808_preliminary.html').read_bytes()
issuer_warning={'url':'https://nvidianews.nvidia.com/news/nvidia-announces-preliminary-financial-resultsfor-second-quarter-fiscal-2023','sha256':hashlib.sha256(issuer_raw).hexdigest(),'published_date':'2022-08-08','preliminary_revenue_million_usd':6700,'prior_outlook_million_usd':8100,'later_consensus_million_usd':float(events.loc[(events.symbol=='NVDA') & (events.report_date=='2022-08-24'),'consensus_revenue_million_usd'].iloc[0]),'interpretation':'later newsletter retains consensus above already-published warning; do not label repeated bad news as novel on actual earnings date'}
audit['revenue_estimate_available_rows']=int(events.consensus_revenue_million_usd.notna().sum())
audit['smart_estimate_available_rows']=int(events.smart_eps.notna().sum())
audit['preannouncement_control_required']=True
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],recorder_name='public_ibes_newsletter_2022_feasibility',uri=str(tracking)):
    rec=R.get_recorder();rec.log_params(plan_sha256=hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),model_fits=0,backtests=0)
    rec.save_objects(artifact_path='sources',manifest=manifest,pdfs={p.name:p.read_bytes() for p in raw.glob('*.pdf')},parser=(root/'src/data/refinitiv_newsletter.py').read_text(),issuer_preliminary_html=issuer_raw,issuer_warning_audit=issuer_warning)
    rec.save_objects(artifact_path='analysis',events=events,audit=audit,calendar=calendar)
    rec.log_metrics(source_events=len(events),registered_artifact_count=8)
summary={'experiment':plan['experiment'],'decision':'source_feasible_needs_extended_coverage_and_publication_evidence' if len(events)==len(plan['events']) else 'needs_source_repair','source_audit':audit,'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/str(rec.experiment_id)/rec.id),'provider_uri':str(root/plan['price_provider']),'region':'us','as_of':'2026-10-09','qlib_docs':'https://qlib.readthedocs.io/en/latest/component/workflow.html','objective_achieved':False,'goal_remains_active':True,'promotion_eligible':False,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['sources/*','analysis/*']}}
artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';verification={'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size} for p in sorted(artifacts.rglob('*')) if p.is_file()]}
(card/'verification_manifest.json').write_text(json.dumps(verification,indent=2)+'\n')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id))
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':plan['experiment'],'provider_uri':summary['provider_uri'],'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']}));events.to_csv(card/'source_inventory.csv',index=False)
audit


{'expected_events': 12,
 'parsed_events': 12,
 'failures': [],
 'per_symbol': {'AMD': 4, 'INTC': 4, 'NVDA': 4},
 'prior_day_document_clock_rows': 11,
 'transport_certificate_verified_rows': 0,
 'publication_clock_certified': False,
 'point_in_time_consensus_certified': False,
 'model_fits': 0,
 'backtests': 0,
 'revenue_estimate_available_rows': 11,
 'smart_estimate_available_rows': 11,
 'preannouncement_control_required': True}

# 免费历史一致预期来源可行性

固定2022年NVDA、AMD、INTC全部12次财报日历，成功恢复12条EPS普通一致预期，各公司4条；其中11条同时含StarMine SmartEstimate与营收预期。未看事件收益，0拟合、0回测，不能声称新增超额。

来源为Refinitiv自行发布的带日期PDF简报。表格实际列顺序为SmartEstimate、普通EPS预期、去年EPS、营收预期；已通过原始版面核对，不能依赖PDF抽取后的错序表头。11份文件生成/修改时间早于财报日；AMD2022-08-02的DayAhead返回404，改用同供应商当天07:23修改的Morning News Call，只有EPS，其他字段保留未知。NTS报告时间保留未知，不当作具体时刻。初次解析失败、跨页/大小写修复及404记录均保留在qualification_attempts.json。

发现具体的事件重复风险：NVDA2022-08-24的简报仍保留8101.61百万美元营收预期，而公司8月8日已发布6700百万美元初步营收。这是可靠的“预期快照”不一定等于新鲜市场预期的实例；必须把公司预警纳入事件时间线，不能在正式财报日将已公开的信息再算一次新冲击。原始公司公告及核对结果已保存Recorder，不修改供应商数值。

传输与时间限制：正常下载遇到公开旧域名证书过期，仅对无凭证固定公共文件做单次证书例外读取，未修改全局TLS。文件SHA可保证本次冻结后不变，但不等于原站传输身份验证或历史首次公开认证；PDF生成时间也不能单独证明当时网页可访问。所有publication_clock_certified和promotion_eligible均为false。

决定为source_feasible_needs_extended_coverage_and_publication_evidence。下一步沿固定财报日历扩展覆盖，补公司原始实际值与提前预警时间，保持普通预期和SmartEstimate分开，不使用最新网站历史值回填，不按事件收益选择报道。形成完整版本/口径/时点合同后才冻结新的信息实验。生产SIM未改，稳定净SOXX超额目标保持active。
